# Batch: reviewed natural questions v2

Run the 20 reviewed original r/NISTControls questions separately from the 136-row benchmark and the 30-row retest. Select **Runtime → Change runtime type → A100 GPU**, then **Run all**. Setup may request Drive access.

The author-reviewed questions, reference labels, protocol and executable fingerprint were published before inference at `71a9a187d0a3fe0d64d71cdefbacdb345cce890d`. Three original-post captures were completed in this version. Keep all 20 questions: three fully answerable from the catalog, 12 partially answerable and five outside its scope. Only two require actual ODP values. These are small exploratory denominators, and the reference labels are not independent expert validation. The launcher creates an isolated Python 3.12.14 environment compatible with the frozen dependencies, independently of the Colab notebook kernel version.

A prior attempt captured five revision-4 contexts, then stopped because the authority check compared raw lowercase/dot-form control IDs against the retriever's normalized IDs. The correction is explicitly registered at `27c437a2176847b6ac82f86b3f1abbb3ba63c1ac`, after that interrupted retrieval and before any answer generation. Questions, labels, models and experimental settings remain unchanged. Use a separate `natural_questions_v2_authority_fix` result directory and retain the original attempt. See `EXECUTION_REPAIR.json`.

Both answer paths share the pinned Qwen2.5-7B-Instruct BF16 model and matched evidence. Outputs, all generation attempts, runtime identity and logs are retained in Drive. Automatic completion is followed by a separate author review of 40 question/path pairs.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import os
import platform
import shutil
import subprocess
import sys
from google.colab import drive

drive.mount('/content/drive')
SOURCE_COMMIT = '27c437a2176847b6ac82f86b3f1abbb3ba63c1ac'
REPO_DIR = Path('/content/ComplianceGPT_natural_v2')
STUDY_RELATIVE = Path('experiments/external_validity/natural_questions_v2')
OUTPUT_DIR = Path('/content/drive/MyDrive/ComplianceGPT/experiments/natural_questions_v2_authority_fix')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# Launcher logs must not be mistaken for outcomes before run_config.json exists.
LAUNCHER_LOG_DIR = OUTPUT_DIR.with_name(OUTPUT_DIR.name + '_launcher_logs')
LAUNCHER_LOG_DIR.mkdir(parents=True, exist_ok=True)
CONSOLE_LOG = LAUNCHER_LOG_DIR / 'console.log'
LAUNCHER_LOG = LAUNCHER_LOG_DIR / 'launcher_steps.jsonl'
test_environment = os.environ.copy()
test_environment.update(USE_TORCH='1', USE_TF='0', USE_FLAX='0',
                        TOKENIZERS_PARALLELISM='false', HF_HUB_DISABLE_TELEMETRY='1',
                        CUBLAS_WORKSPACE_CONFIG=':4096:8')
test_environment['PYTHONPATH'] = str(REPO_DIR / 'src') + ':' + str(REPO_DIR)

def digest(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def run_logged(command, cwd=None):
    event = {'command': list(command), 'cwd': str(cwd) if cwd else None,
             'started_at_utc': datetime.now(timezone.utc).isoformat()}
    return_code = None
    try:
        with CONSOLE_LOG.open('a', encoding='utf-8') as log:
            log.write('\nAttempt: ' + event['started_at_utc'] + '\n' + repr(list(command)) + '\n')
            log.flush()
            with subprocess.Popen(command, cwd=cwd, env=test_environment,
                                  stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                  text=True, bufsize=1) as process:
                for line in process.stdout:
                    print(line, end='')
                    log.write(line)
                    log.flush()
                return_code = process.wait()
        if return_code:
            raise RuntimeError('Command failed. The Drive console log and existing checkpoints are retained. Review the failure before resuming.')
    finally:
        event.update(return_code=return_code, finished_at_utc=datetime.now(timezone.utc).isoformat())
        with LAUNCHER_LOG.open('a', encoding='utf-8') as stream:
            stream.write(json.dumps(event, ensure_ascii=False) + '\n')
            stream.flush()
            os.fsync(stream.fileno())

print('Results and checkpoints:', OUTPUT_DIR)
print('Python:', platform.python_version())


In [ ]:
if not REPO_DIR.exists():
    run_logged(['git', 'clone', '--filter=blob:none', '--no-checkout',
                'https://github.com/Wenjia1215/ComplianceGPT.git', str(REPO_DIR)])
    run_logged(['git', 'fetch', '--depth=1', 'origin', SOURCE_COMMIT], cwd=REPO_DIR)
    run_logged(['git', 'checkout', '--detach', SOURCE_COMMIT], cwd=REPO_DIR)
head = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
if head != SOURCE_COMMIT:
    raise RuntimeError('This working directory has a different commit. Use a fresh Colab runtime; preserve the existing Drive result folder.')
if subprocess.check_output(['git', 'status', '--porcelain', '--untracked-files=no'], cwd=REPO_DIR, text=True):
    raise RuntimeError('The registered checkout contains modified source files.')
run_logged(['git', 'show', '-s', '--format=fuller', 'HEAD'], cwd=REPO_DIR)
run_logged([sys.executable, '-u', str(STUDY_RELATIVE / 'run_natural_questions.py'),
            '--output-dir', str(OUTPUT_DIR), '--preflight-only'], cwd=REPO_DIR)
print('Exact public question/label/code registration passed, before model loading.')


In [ ]:
run_logged(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'])
# NumPy 2.0.2 supports Python through 3.12. Keep its frozen version instead
# of changing the scientific stack to accommodate a newer Colab kernel.
# https://github.com/numpy/numpy/releases/tag/v2.0.2
run_logged([sys.executable, '-m', 'pip', 'install', '-q', 'uv==0.12.19'])
UV_COMMAND = [sys.executable, '-m', 'uv']
RUNTIME_DIR = Path('/content/ComplianceGPT_natural_v2_py312')
EXPERIMENT_PYTHON = str(RUNTIME_DIR / 'bin/python')
if not Path(EXPERIMENT_PYTHON).exists():
    run_logged(UV_COMMAND + ['venv', '--python', '3.12.14', '--seed', str(RUNTIME_DIR)])
actual_python = subprocess.check_output([EXPERIMENT_PYTHON, '-c',
                                        'import platform; print(platform.python_version())'],
                                       env=test_environment, text=True).strip()
if actual_python != '3.12.14':
    raise RuntimeError('The study interpreter must be Python 3.12.14. Use a fresh Colab runtime.')
# The new environment has no Colab media or TensorFlow integrations.
run_logged([EXPERIMENT_PYTHON, '-m', 'pip', 'install', '-q', 'torch==2.6.0',
            '--index-url', 'https://download.pytorch.org/whl/cu124'])
run_logged([EXPERIMENT_PYTHON, '-m', 'pip', 'install', '-q', '-r',
            str(REPO_DIR / STUDY_RELATIVE / 'requirements-colab.txt')])
run_logged([EXPERIMENT_PYTHON, '-m', 'pip', 'freeze'])
print('Pinned text-model stack installed in the isolated Python', actual_python, 'environment.')


In [ ]:
model_preflight = '''
import importlib.util
import torch
unexpected = [name for name in ('timm', 'torchvision', 'torchaudio') if importlib.util.find_spec(name)]
if unexpected:
    raise RuntimeError('Optional media backends remain: ' + ', '.join(unexpected))
if not torch.cuda.is_available() or not torch.cuda.is_bf16_supported():
    raise RuntimeError('Select a CUDA GPU supporting BF16, preferably A100.')
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoModel, AutoModelForSequenceClassification, BertConfig, XLMRobertaConfig
common = dict(vocab_size=32, hidden_size=8, num_hidden_layers=1,
              num_attention_heads=2, intermediate_size=16)
AutoModel.from_config(BertConfig(**common))
AutoModelForSequenceClassification.from_config(XLMRobertaConfig(**common))
print('Text-model imports and tiny loading paths passed.')
print('Actual GPU:', torch.cuda.get_device_name(0), 'Torch:', torch.__version__, 'CUDA:', torch.version.cuda)
'''
run_logged([EXPERIMENT_PYTHON, '-c', model_preflight], cwd=REPO_DIR)
run_logged([EXPERIMENT_PYTHON, '-m', 'unittest', 'tests.test_natural_questions',
            'tests.test_natural_questions_v2', 'tests.test_evidence_window',
            'tests.test_natural_questions_launcher', 'tests.test_natural_questions_authority', '-q'], cwd=REPO_DIR)
print('Offline review, scoring and shared-window checks passed. No study inference yet.')


The next cell performs the registered experiment. Keep the original questions and frozen labels unchanged. A failed attempt retains logs and per-question checkpoints. Resume only with this same source commit and compatible recorded runtime. Do not delete failed calls or rerun to select favorable outputs.


In [ ]:
REPAIRED_SOURCE_COMMIT = '27c437a2176847b6ac82f86b3f1abbb3ba63c1ac'
if SOURCE_COMMIT != REPAIRED_SOURCE_COMMIT:
    # Recover an already prepared session without changing its dependencies.
    ORIGINAL_SOURCE_COMMIT = '71a9a187d0a3fe0d64d71cdefbacdb345cce890d'
    if SOURCE_COMMIT != ORIGINAL_SOURCE_COMMIT or OUTPUT_DIR.name != 'natural_questions_v2':
        raise RuntimeError('Unexpected prior study identity. All files have been left unchanged.')
    current_head = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
    if current_head not in {ORIGINAL_SOURCE_COMMIT, REPAIRED_SOURCE_COMMIT}:
        raise RuntimeError('Unexpected checkout. Preserve this session for review.')
    if subprocess.check_output(['git', 'status', '--porcelain', '--untracked-files=no'],
                               cwd=REPO_DIR, text=True):
        raise RuntimeError('The checkout contains modified source files. They have been left unchanged.')
    prior_config_path = OUTPUT_DIR / 'run_config.json'
    prior_config = json.loads(prior_config_path.read_text())
    if (prior_config['signature']['source_commit'] != ORIGINAL_SOURCE_COMMIT
            or prior_config['signature']['registration_sha256'] !=
            'ad349c068c90ea366f5ee0817ea71b8670585adca3c7353914eb82eb0e259f20'):
        raise RuntimeError('The prior checkpoint belongs to another registration. It has been left unchanged.')
    generation_markers = [OUTPUT_DIR / name for name in
                          ('generation_calls.jsonl', 'run_manifests.json', 'paired_identity_audit.json',
                           'per_question_outcomes.jsonl', 'summary.json')]
    if any(p.exists() for p in generation_markers) or any((OUTPUT_DIR / 'contracts').rglob('*.csv')):
        raise RuntimeError('The prior attempt contains generation artifacts. Preserve it for audit before restarting.')
    PRIOR_ATTEMPT_DIR = OUTPUT_DIR.resolve()
    PRIOR_ATTEMPT_LOG_DIR = OUTPUT_DIR.with_name(OUTPUT_DIR.name + '_launcher_logs')
    prior_hashes = {str(p.relative_to(PRIOR_ATTEMPT_DIR)): digest(p)
                    for p in sorted(PRIOR_ATTEMPT_DIR.rglob('*')) if p.is_file()}
    repaired_output = OUTPUT_DIR.with_name('natural_questions_v2_authority_fix')
    repair_log_dir = repaired_output.with_name(repaired_output.name + '_launcher_logs')
    repair_log_dir.mkdir(parents=True, exist_ok=True)
    CONSOLE_LOG = repair_log_dir / 'console.log'
    LAUNCHER_LOG = repair_log_dir / 'launcher_steps.jsonl'
    if current_head != REPAIRED_SOURCE_COMMIT:
        run_logged(['git', 'fetch', '--depth=1', 'origin', REPAIRED_SOURCE_COMMIT], cwd=REPO_DIR)
        run_logged(['git', 'checkout', '--detach', REPAIRED_SOURCE_COMMIT], cwd=REPO_DIR)
    if subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip() != REPAIRED_SOURCE_COMMIT:
        raise RuntimeError('The repaired source checkout did not complete.')
    SOURCE_COMMIT = REPAIRED_SOURCE_COMMIT
    OUTPUT_DIR = repaired_output
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    (OUTPUT_DIR / 'prior_attempt_provenance.json').write_text(json.dumps({
        'original_source_commit': ORIGINAL_SOURCE_COMMIT,
        'repaired_source_commit': SOURCE_COMMIT,
        'original_result_directory': str(PRIOR_ATTEMPT_DIR),
        'original_launcher_log_directory': str(PRIOR_ATTEMPT_LOG_DIR),
        'no_prior_generation_artifacts': True, 'file_sha256': prior_hashes
    }, indent=2, sort_keys=True) + '\n')
    run_logged([EXPERIMENT_PYTHON, '-u', str(STUDY_RELATIVE / 'run_natural_questions.py'),
                '--output-dir', str(OUTPUT_DIR), '--preflight-only'], cwd=REPO_DIR)
    run_logged([EXPERIMENT_PYTHON, '-m', 'unittest', 'tests.test_natural_questions',
                'tests.test_natural_questions_v2', 'tests.test_evidence_window',
                'tests.test_natural_questions_launcher', 'tests.test_natural_questions_authority', '-q'], cwd=REPO_DIR)
    print('Prior retrieval attempt retained:', PRIOR_ATTEMPT_DIR)
    print('Repaired registered execution:', SOURCE_COMMIT, OUTPUT_DIR)

# This cell also repairs the log layout in an already prepared Colab session.
# The runner's outcome/runtime compatibility checks stay unchanged.
from uuid import uuid4
LAUNCHER_LOG_DIR = OUTPUT_DIR.with_name(OUTPUT_DIR.name + '_launcher_logs')
LAUNCHER_LOG_DIR.mkdir(parents=True, exist_ok=True)
legacy_log = OUTPUT_DIR / 'launcher_steps.jsonl'
if legacy_log.exists():
    legacy_bytes = legacy_log.read_bytes()
    events = [json.loads(line) for line in legacy_bytes.decode('utf-8').splitlines() if line.strip()]
    event_keys = {'command', 'cwd', 'started_at_utc', 'return_code', 'finished_at_utc'}
    for event in events:
        if not (isinstance(event, dict) and set(event) == event_keys
                and isinstance(event['command'], list) and event['command']
                and all(isinstance(part, str) for part in event['command'])
                and (event['cwd'] is None or isinstance(event['cwd'], str))
                and (event['return_code'] is None or type(event['return_code']) is int)):
            raise RuntimeError('The legacy file is not a launcher log. It has been left unchanged.')
        for key in ('started_at_utc', 'finished_at_utc'):
            if not isinstance(event[key], str) or datetime.fromisoformat(event[key]).utcoffset() is None:
                raise RuntimeError('The legacy launcher log has an invalid timestamp. It has been left unchanged.')
    archived_log = LAUNCHER_LOG_DIR / ('legacy_launcher_steps_' + uuid4().hex + '.jsonl')
    shutil.move(str(legacy_log), str(archived_log))
    assert archived_log.read_bytes() == legacy_bytes
    print('Previous launcher events preserved:', archived_log)
CONSOLE_LOG = LAUNCHER_LOG_DIR / 'console.log'
LAUNCHER_LOG = LAUNCHER_LOG_DIR / 'launcher_steps.jsonl'

run_logged([EXPERIMENT_PYTHON, '-u', str(STUDY_RELATIVE / 'run_natural_questions.py'),
            '--output-dir', str(OUTPUT_DIR)], cwd=REPO_DIR)
# Copy logs into the result package only after the runner has registered this run.
if not (OUTPUT_DIR / 'run_config.json').is_file():
    raise RuntimeError('The runner returned without a run configuration. Logs are retained separately.')
shutil.copytree(LAUNCHER_LOG_DIR, OUTPUT_DIR / 'launcher_logs', dirs_exist_ok=True)
if globals().get('PRIOR_ATTEMPT_DIR') is not None:
    provenance = json.loads((OUTPUT_DIR / 'prior_attempt_provenance.json').read_text())
    for relative, expected in provenance['file_sha256'].items():
        if digest(PRIOR_ATTEMPT_DIR / relative) != expected:
            raise RuntimeError('An earlier attempt file changed. Both runs are retained for review.')
    shutil.copytree(PRIOR_ATTEMPT_DIR, OUTPUT_DIR / 'prior_attempt/result_files', dirs_exist_ok=True)
    if PRIOR_ATTEMPT_LOG_DIR.is_dir():
        shutil.copytree(PRIOR_ATTEMPT_LOG_DIR, OUTPUT_DIR / 'prior_attempt/launcher_logs', dirs_exist_ok=True)
print('Registered generation and automatic scoring finished; qualitative author review remains pending.')


In [ ]:
import csv
summary = json.loads((OUTPUT_DIR / 'summary.json').read_text())
config = json.loads((OUTPUT_DIR / 'run_config.json').read_text())
outcomes = [json.loads(s) for s in (OUTPUT_DIR / 'per_question_outcomes.jsonl').read_text().splitlines() if s.strip()]
assert summary['question_count'] == 20 and summary['paired_generation_count'] == 40
assert len(outcomes) == 40 and len({r['query_id'] for r in outcomes}) == 20
assert summary['scope_counts'] == {'full': 3, 'partial': 12, 'outside': 5}
assert summary['odp_label_counts'] == {'required': 2, 'not_required': 13, 'not_applicable': 5}
assert config['signature']['source_commit'] == SOURCE_COMMIT
assert summary['experiment_complete'] is False
assert summary['independent_correctness_validated'] is False
with (OUTPUT_DIR / 'post_run_semantic_review.csv').open(newline='', encoding='utf-8') as stream:
    assert len(list(csv.DictReader(stream))) == 40
for system in ('compliancegpt', 'generative_baseline'):
    metrics = summary['systems'][system]
    assert metrics['strict_full_catalog_contract']['denominator'] == 3
    assert metrics['odp_author_reference_sensitivity']['denominator'] == 2
    print(system, json.dumps(metrics, indent=2))
print('All 20 paired questions retained. Strict denominator: 3; positive ODP denominator: 2.')
print('The separate scope/responsiveness review must inspect all 40 answers, including actual clarification wording.')

snapshot = OUTPUT_DIR / 'source_snapshot'
registration = json.loads((REPO_DIR / STUDY_RELATIVE / 'registration.json').read_text())
source_paths = {str(STUDY_RELATIVE / name) for name in registration['file_sha256']}
source_paths.update(registration['code_sha256'])
source_paths.update(str(STUDY_RELATIVE / name) for name in ('registration.json', 'PROTOCOL.md', 'README.md', 'QUESTIONS.md'))
for relative in sorted(source_paths):
    target = snapshot / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(REPO_DIR / relative, target)
(OUTPUT_DIR / 'launcher_identity.json').write_text(json.dumps({
    'source_commit': SOURCE_COMMIT, 'kernel_python': platform.python_version(),
    'experiment_python': config['signature']['python'], 'experiment_interpreter': EXPERIMENT_PYTHON,
    'launcher_log_directory': str(LAUNCHER_LOG_DIR),
    'registration_sha256': digest(REPO_DIR / STUDY_RELATIVE / 'registration.json'),
    'automatic_pair_count': 40, 'post_run_author_review': 'pending',
    'final_notebook': 'Save the fully executed notebook from the Colab File menu after Run all finishes.'
}, indent=2) + '\n')


In [ ]:
hashes = {str(p.relative_to(OUTPUT_DIR)): digest(p)
          for p in sorted(OUTPUT_DIR.rglob('*')) if p.is_file() and p.name != 'SHA256SUMS.json'}
(OUTPUT_DIR / 'SHA256SUMS.json').write_text(json.dumps(hashes, indent=2, sort_keys=True) + '\n')
archive_path = Path(shutil.make_archive(str(OUTPUT_DIR.parent / (OUTPUT_DIR.name + '_results')),
                                      'zip', root_dir=OUTPUT_DIR))
print('Result package:', archive_path)
print('Package SHA-256:', digest(archive_path))
print('Return this ZIP plus the fully executed notebook (File → Download → Download .ipynb).')
print('Keep all cell outputs. The returned notebook will be archived with the results in GitHub.')
from google.colab import files
files.download(str(archive_path))


After execution, download this notebook with all cell outputs preserved and return it with the result ZIP. The next review checks source/model/context identities, paired completeness, actual responses and unsupported claims before dissertation integration. The experiment is not complete merely because automatic scoring succeeded.
